# Step 3: Section Reranking, Multimodal Answer Generation & Phase 0 Failure Analysis

This notebook implements **OMGM's Stage 3 Baseline Pipeline** and executes the **Phase 0 Failure Analysis** for the *Agentic Retrieval Controller* research project:
1. **Input Ingestion:** Loads Stage 2 reranking results (`result_step_2.json`), 100K Wikipedia Knowledge Base (`wiki_100_dict_v4.json`), and the 1,000 InfoSeek test queries.
2. **Step 3.1 — Section Reranking (BGE Cross-Encoder):** For each Top-1 entity selected in Stage 2, reconstructs its Wikipedia article sections, computes query-to-section relevance with `BAAI/bge-reranker-v2-m3`, and fuses it with Stage 2 fine-grained similarity ($eta = 0.2$) to select the single best evidence section.
3. **Step 3.2 — Multimodal/LLM Answer Generation:** Feeds `(Query Image, Question, Top-1 Section)` into the downstream generator (`llava-hf/llava-1.5-7b-hf` in 4-bit quantization, with optional toggle for `LLaMA-3-8B` / `Mistral-7B`).
4. **Step 3.3 — Evaluation & Phase 0 Failure Categorization:** Evaluates exact-match and numerical accuracy against InfoSeek ground truth, and automatically partitions all 1,000 queries into:
   - **Successes:** Correct VQA predictions.
   - **Category A:** Correct entity missed in Stage 1 Top-20 (Retrieval ceiling failure).
   - **Category B:** Correct entity was in Top-20, but Stage 2 reranker chose the wrong entity as Top-1 (Reranker confusion).
   - **Category C:** Correct entity chosen as Top-1, but Step 3 BGE chose the wrong section (Section selection failure).
   - **Category D:** Correct entity & correct section, but generator hallucinated or gave wrong format (Generation failure).


### Cell 1: Environment Verification & GPU Diagnostics


In [1]:
import os
import sys
import gc
import glob
import json
import time
import torch
import PIL.Image
from tqdm import tqdm

print("=" * 60)
print("ENVIRONMENT & GPU DIAGNOSTICS")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total GPU VRAM:   {total_mem:.2f} GB")
else:
    print("WARNING: CUDA is NOT available. Stage 3 requires a GPU (Tesla T4 or better).")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("=" * 60)


ENVIRONMENT & GPU DIAGNOSTICS
Python Version:   3.13.15
PyTorch Version:  2.14.1+cu130
CUDA Available:   True
Device Name:      Tesla T4
Total GPU VRAM:   14.56 GB


### Cell 2: Dependencies Installation
Installs lightweight requirements including `bitsandbytes` and `accelerate` for 4-bit quantized generation.


In [2]:
# Install rock-solid stable transformers, accelerate, bitsandbytes
# Note: We do NOT upgrade torchvision to avoid breaking pre-installed PyTorch
!pip install -q "transformers==4.46.3" accelerate bitsandbytes sentencepiece protobuf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 108.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 20.9 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


### Cell 3: Path Configuration & Dataset Autodetection
Configures paths to Stage 2 `result_step_2.json`, Knowledge Base, 1,000 query sample CSV, and query images.


In [3]:
# ---------------------------------------------------------------------------
# Kaggle Dataset & Path Configuration for Stage 3
# ---------------------------------------------------------------------------
import os
import glob

BASE_DS_DIR = "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds"
ALT_DS_DIR = "/kaggle/input/agentic-vqa-ds"

def resolve_path(exact_path, search_patterns=None, is_dir=False):
    """Helper to resolve paths with primary check and fallback glob search."""
    if os.path.exists(exact_path):
        if is_dir and os.path.isdir(exact_path):
            return exact_path
        elif not is_dir and os.path.isfile(exact_path):
            return exact_path
        elif not is_dir and os.path.exists(exact_path):
            return exact_path

    alt_path = exact_path.replace(BASE_DS_DIR, ALT_DS_DIR)
    if os.path.exists(alt_path):
        return alt_path

    if search_patterns:
        for pat in search_patterns:
            matches = glob.glob(pat, recursive=True)
            for m in matches:
                if is_dir and os.path.isdir(m):
                    return m
                elif not is_dir and os.path.isfile(m):
                    return m
                elif os.path.exists(m):
                    return m
    return exact_path

# 1. Stage 2 Retrieval Results JSON (result_step_2.json)
STAGE2_RESULT_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result_step_2.json",
    search_patterns=[
        "/kaggle/input/**/result_step_2.json",
        "/kaggle/input/**/result.json",
        "/kaggle/working/step2_results/result.json",
        "/kaggle/working/**/result*.json",
    ],
    is_dir=False
)

# 2. Wikipedia 100k Knowledge Base
KB_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json",
    search_patterns=[
        "/kaggle/input/**/wiki_100_dict_v4.json",
        "/kaggle/input/**/infoseek_100k_wiki/*.json",
        "/kaggle/input/**/wiki_100k*.json",
    ],
    is_dir=False
)

# 3. Query Images Directory (1,000 InfoSeek test images)
QUERY_IMG_DIR = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images",
    search_patterns=[
        "/kaggle/input/**/infoseek_val_1000_images/infoseek_val_1000_images",
        "/kaggle/input/**/infoseek_val_1000_images",
        "/kaggle/input/**/images",
    ],
    is_dir=True
)

# 4. 1,000 Sampled Test Questions CSV (infoseek_test_sample_1000.csv)
SAMPLE_FILE = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_test_sample_1000.csv",
    search_patterns=[
        "/kaggle/input/**/infoseek_test_sample_1000.csv",
        "/kaggle/input/**/infoseek_test*.csv",
        "/kaggle/input/**/test.csv",
    ],
    is_dir=False
)

# 5. Output Directory for Stage 3
OUTPUT_DIR = "/kaggle/working/step3_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
STEP2_BETA = 0.2  # Fusion weight between Step 2 multimodal score and Step 3 BGE score (OMGM default)

print("=" * 60)
print("STAGE 3 PIPELINE PATHS CONFIGURED")
print("=" * 60)
print(f"1. Stage 2 Result JSON: {STAGE2_RESULT_PATH} (Exists: {os.path.exists(str(STAGE2_RESULT_PATH))})")
print(f"2. Knowledge Base JSON: {KB_PATH} (Exists: {os.path.exists(str(KB_PATH))})")
print(f"3. Query Images Dir:    {QUERY_IMG_DIR} (Exists: {os.path.isdir(str(QUERY_IMG_DIR))})")
print(f"4. Sample Questions:    {SAMPLE_FILE} (Exists: {os.path.exists(str(SAMPLE_FILE))})")
print(f"   Output Directory:    {OUTPUT_DIR}")
print(f"   Step2 Beta Weight:   {STEP2_BETA}")
print("=" * 60)


STAGE 3 PIPELINE PATHS CONFIGURED
1. Stage 2 Result JSON: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result_step_2.json (Exists: True)
2. Knowledge Base JSON: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json (Exists: True)
3. Query Images Dir:    /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images (Exists: True)
4. Sample Questions:    /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_test_sample_1000.csv (Exists: True)
   Output Directory:    /kaggle/working/step3_results
   Step2 Beta Weight:   0.2


### Cell 4: Load Knowledge Base & Stage 2 Results
Loads the 100K Wikipedia knowledge base and matches each query with its Top-1 entity from Stage 2.


In [4]:
import csv
import json

class WikipediaKnowledgeBaseEntry:
    """Represents a single Wikipedia article entry in the knowledge base."""
    def __init__(self, entry_dict):
        self.title = entry_dict.get("title", "")
        self.url = entry_dict.get("url", "")
        self.image_urls = entry_dict.get("image_urls", [])
        self.image_reference_descriptions = entry_dict.get("image_reference_descriptions", [])
        self.image_section_indices = entry_dict.get("image_section_indices", [])
        self.section_titles = entry_dict.get("section_titles", [])
        self.section_texts = entry_dict.get("section_texts", [])

def reconstruct_wiki_sections(knowledge_entry):
    """Reconstruct sections from knowledge base entry, matching OMGM's section formatting."""
    title = knowledge_entry.title
    sections = []
    for it, section_title in enumerate(knowledge_entry.section_titles):
        sec_lower = section_title.lower()
        if "external links" in sec_lower or "references" in sec_lower:
            continue
        section_text = knowledge_entry.section_texts[it] if it < len(knowledge_entry.section_texts) else ""
        sec_str = f"# Wiki Article: {title}\n## Section Title: {section_title}\n{section_text}"
        sections.append(sec_str)
    return sections

print(f"Loading Knowledge Base from {KB_PATH}...")
with open(KB_PATH, "r", encoding="utf-8") as f:
    kb_dict = json.load(f)
knowledge_base_dict = {entry["url"]: WikipediaKnowledgeBaseEntry(entry) for entry in kb_dict.values()}
print(f"Loaded Knowledge Base with {len(knowledge_base_dict):,} entries.")
del kb_dict
gc.collect()

print(f"\nLoading Stage 2 results from {STAGE2_RESULT_PATH}...")
with open(STAGE2_RESULT_PATH, "r", encoding="utf-8") as f:
    stage2_results = json.load(f)
print(f"Loaded Stage 2 outputs for {len(stage2_results):,} queries.")

# Load ground-truth test sample manifest
sample_data = {}
qid_to_img_id = {}
if SAMPLE_FILE and os.path.exists(SAMPLE_FILE):
    with open(SAMPLE_FILE, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            did = row.get("data_id")
            if not did:
                continue
            did = str(did).strip()
            sample_data[did] = row
            img_id = row.get("dataset_image_ids") or row.get("image_id") or row.get("dataset_image_id")
            if img_id:
                qid_to_img_id[did] = str(img_id).split("|")[0].strip()
print(f"Loaded metadata for {len(sample_data):,} queries from sample CSV.")

def get_query_image_path(query_key, search_dir, qid_map=None):
    """Resolves local file path of a query image."""
    candidates = []
    if qid_map and str(query_key) in qid_map:
        candidates.append(qid_map[str(query_key)])
    clean_k = str(query_key).split("|")[0].strip()
    candidates.append(clean_k)
    
    for raw_id in candidates:
        clean_id = raw_id.strip()
        for ext in ["", ".jpg", ".JPEG", ".png", ".JPG", ".PNG"]:
            p = os.path.join(search_dir, f"{clean_id}{ext}")
            if os.path.isfile(p):
                return p
                
    for raw_id in candidates:
        matches = glob.glob(os.path.join(search_dir, f"*{raw_id}*"))
        for m in matches:
            if os.path.isfile(m) and m.lower().endswith((".jpg", ".jpeg", ".png", ".JPEG", ".PNG")):
                return m
                
    return None


Loading Knowledge Base from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json...
Loaded Knowledge Base with 100,000 entries.

Loading Stage 2 results from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result_step_2.json...
Loaded Stage 2 outputs for 1,000 queries.
Loaded metadata for 1,000 queries from sample CSV.


### Cell 5: Pass 1 — Section Reranking (BGE Cross-Encoder)
Uses `BAAI/bge-reranker-v2-m3` to compute query-to-section relevance and fuses it with Stage 2 fine-grained similarity ($eta = 0.2$). Then frees GPU VRAM completely before the generator is loaded.


In [5]:
# ---------------------------------------------------------------------------
# Pass 1: Section Reranking with BAAI/bge-reranker-v2-m3
# ---------------------------------------------------------------------------
import os
import json
import gc
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SECTIONS_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "step3_reranked_sections.json")

if os.path.exists(SECTIONS_OUTPUT_FILE):
    print(f"Found existing section reranking file at {SECTIONS_OUTPUT_FILE}. Loading...")
    with open(SECTIONS_OUTPUT_FILE, "r", encoding="utf-8") as f:
        reranked_sections = json.load(f)
    print(f"Loaded {len(reranked_sections):,} pre-computed top-1 sections.")
else:
    print("Loading BAAI/bge-reranker-v2-m3 cross-encoder model...")
    reranker_name = "BAAI/bge-reranker-v2-m3"
    bge_tokenizer = AutoTokenizer.from_pretrained(reranker_name)
    bge_model = AutoModelForSequenceClassification.from_pretrained(
        reranker_name,
        torch_dtype=torch.float16
    ).to(device).eval()
    print("BGE Section Reranker loaded on GPU.")

    reranked_sections = {}
    q_ids = list(stage2_results.keys())

    print(f"\nReranking sections for all {len(q_ids):,} queries...")
    with torch.no_grad():
        for q_id in tqdm(q_ids, desc="BGE Section Reranking"):
            q_res = stage2_results[q_id]
            question = q_res.get("question", "")
            top1_url = q_res.get("top1_entity_url", "")
            
            # If top1_entity not in KB, fallback to empty
            if top1_url not in knowledge_base_dict:
                reranked_sections[q_id] = {
                    "top1_section": "",
                    "all_sections_count": 0,
                    "fusion_score": 0.0
                }
                continue
                
            kb_entry = knowledge_base_dict[top1_url]
            section_list = reconstruct_wiki_sections(kb_entry)
            
            if len(section_list) == 0:
                reranked_sections[q_id] = {
                    "top1_section": "",
                    "all_sections_count": 0,
                    "fusion_score": 0.0
                }
                continue
                
            # Score question vs all candidate sections in mini-batches
            pairs = [[question, s] for s in section_list]
            batch_size = 4
            all_bge_scores = []
            
            for b_idx in range(0, len(pairs), batch_size):
                b_pairs = pairs[b_idx : b_idx + batch_size]
                inputs = bge_tokenizer(
                    b_pairs,
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=4096
                ).to(device)
                logits = bge_model(**inputs, return_dict=True).logits.view(-1).float()
                all_bge_scores.append(logits)
                
            bge_raw_scores = torch.cat(all_bge_scores, dim=0)
            q2s_scores = torch.sigmoid(bge_raw_scores)
            
            # Retrieve Step 2 section similarity vector
            step2_top1_sec_sim = q_res.get("sec_sim", [[]])[0]
            
            if len(step2_top1_sec_sim) == len(q2s_scores):
                # Full score fusion per OMGM paper
                step2_sim_t = torch.tensor(step2_top1_sec_sim, device=device, dtype=torch.float32)
                fused_scores = STEP2_BETA * step2_sim_t + (1.0 - STEP2_BETA) * q2s_scores
            else:
                # Fallback to pure BGE if dimensions mismatch
                fused_scores = q2s_scores
                
            best_idx = torch.argmax(fused_scores).item()
            best_section = section_list[best_idx]
            
            reranked_sections[q_id] = {
                "top1_section": best_section,
                "all_sections_count": len(section_list),
                "best_section_index": best_idx,
                "fusion_score": float(fused_scores[best_idx].item())
            }

    with open(SECTIONS_OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(reranked_sections, f, indent=2)
    print(f"Saved reranked sections to {SECTIONS_OUTPUT_FILE}")

    # Unload BGE model and free 100% of GPU VRAM
    del bge_model
    del bge_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Unloaded BGE Reranker. GPU memory cleared for Generator.")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading BAAI/bge-reranker-v2-m3 cross-encoder model...


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

BGE Section Reranker loaded on GPU.

Reranking sections for all 1,000 queries...


BGE Section Reranking: 100%|██████████| 1000/1000 [05:08<00:00,  3.24it/s]


Saved reranked sections to /kaggle/working/step3_results/step3_reranked_sections.json
Unloaded BGE Reranker. GPU memory cleared for Generator.


### Cell 6: Pass 2 — Generator Initialization
Loads the primary MLLM generator (`llava-hf/llava-1.5-7b-hf`) in 4-bit precision via `BitsAndBytesConfig`, ensuring it fits comfortably on Kaggle's 15 GB Tesla T4 GPU.


In [6]:
# ---------------------------------------------------------------------------
# Pass 2: Generator Model Initialization
# ---------------------------------------------------------------------------
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Choose generator model: 'llava1_5' (MLLM - Best in OMGM paper) or 'llama3' (LLM) or 'mistral'
GENERATOR_TYPE = "llava1_5"  # Options: 'llava1_5', 'llama3', 'mistral'

# 4-bit Quantization configuration for Kaggle Tesla T4 (15 GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# OMGM Official InfoSeek VLM Instruction (Appendix A, Table 17)
VLM_INFOSEEK_INSTRUCTION = (
    "Answer the encyclopedic question about the given image. "
    "Don't mention the visual content of image in your output. "
    "Directly output the answer of the question according to the Context.\n"
    "If you need to answer questions about numbers or time, please output the corresponding numerical format directly.\n"
    "If the context does not contain the information required to answer the question, you should answer the question using internal model knowledge.\n"
    "There is an example:\n"
    "- Context: # Wiki Article: Dolomites\n## Section Title: Dolomites\n"
    "The Dolomites, also known as the Dolomite Mountains, Dolomite Alps or Dolomitic Alps, are a mountain range located in northeastern Italy. "
    "The Dolomites are located in the regions of Veneto, Trentino-Alto Adige/Südtirol and Friuli Venezia Giulia, covering an area shared between the provinces of Belluno, Vicenza, Verona, Trentino, South Tyrol, Udine and Pordenone.\n"
    "- Question: Which city or region does this mountain locate in?\n"
    "Just answer the questions , no explanations needed. Short answer is: Province of Belluno\n\n"
)

def build_vqa_prompt(question, section_text, max_chars=3000):
    """Constructs the OMGM prompt with context length trimming."""
    if len(section_text) > max_chars:
        section_text = section_text[:max_chars] + "..."
    prompt = (
        f"- Context: {section_text}\n"
        f"- Question: {question}\n"
        f"Just answer the questions , no explanations needed. Short answer is:"
    )
    return prompt

print(f"Initializing Generator: {GENERATOR_TYPE} with 4-bit Quantization...")
if GENERATOR_TYPE == "llava1_5":
    mllm_model_id = "llava-hf/llava-1.5-7b-hf"
    processor = AutoProcessor.from_pretrained(mllm_model_id)
    generator_model = LlavaForConditionalGeneration.from_pretrained(
        mllm_model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).eval()
    print("LLaVA-1.5-7B loaded successfully in 4-bit.")
elif GENERATOR_TYPE == "llama3":
    llm_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
    generator_model = AutoModelForCausalLM.from_pretrained(
        llm_model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).eval()
    print("LLaMA-3-8B loaded successfully in 4-bit.")
elif GENERATOR_TYPE == "mistral":
    llm_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
    generator_model = AutoModelForCausalLM.from_pretrained(
        llm_model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).eval()
    print("Mistral-7B loaded successfully in 4-bit.")

allocated_mem = torch.cuda.memory_allocated() / (1024**3)
print(f"GPU VRAM currently allocated: {allocated_mem:.2f} GB")


Initializing Generator: llava1_5 with 4-bit Quantization...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA-1.5-7B loaded successfully in 4-bit.
GPU VRAM currently allocated: 1.60 GB


### Cell 7: Pilot Test on 5 Queries
Verifies inference functionality and output format on a small 5-query slice before starting full generation.


In [7]:
# ---------------------------------------------------------------------------
# Pilot Test on 5 Queries
# ---------------------------------------------------------------------------
pilot_qids = list(stage2_results.keys())[:5]

print("Running Pilot Test on 5 queries:")
print("-" * 60)

for idx, qid in enumerate(pilot_qids):
    q_data = stage2_results[qid]
    question = q_data.get("question", "")
    top1_sec = reranked_sections[qid].get("top1_section", "")
    gt_ans = sample_data.get(qid, {}).get("answer", "N/A")
    
    # Load query image
    img_path = get_query_image_path(qid, QUERY_IMG_DIR, qid_to_img_id)
    if img_path and os.path.isfile(img_path):
        query_img = PIL.Image.open(img_path).convert("RGB")
    else:
        query_img = PIL.Image.new("RGB", (224, 224), color=(0, 0, 0))
        
    prompt_text = VLM_INFOSEEK_INSTRUCTION + build_vqa_prompt(question, top1_sec)
    
    if GENERATOR_TYPE == "llava1_5":
        formatted_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor(text=formatted_prompt, images=query_img, return_tensors="pt").to(device)
        with torch.no_grad():
            output_tokens = generator_model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False
            )
        pred_text = processor.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    else:
        # LLM text-only generation
        messages = [
            {"role": "user", "content": prompt_text}
        ]
        enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
        with torch.no_grad():
            output_tokens = generator_model.generate(
                enc,
                max_new_tokens=32,
                do_sample=False
            )
        pred_text = tokenizer.decode(output_tokens[0][enc.shape[1]:], skip_special_tokens=True).strip()
        
    print(f"Query [{idx+1}/5] ID:       {qid}")
    print(f"Question:             {question}")
    print(f"Top-1 Entity:         {q_data.get('top1_entity_title')}")
    print(f"Ground Truth Answer:  {gt_ans}")
    print(f"Predicted Answer:     {pred_text}")
    print("-" * 60)


Running Pilot Test on 5 queries:
------------------------------------------------------------


Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.


Query [1/5] ID:       infoseek_val_00000053
Question:             Who was this building unveiled by?
Top-1 Entity:         Canadian National Vimy Memorial
Ground Truth Answer:  Edward Albert Christian George Andrew Patrick David|David Windsor|Edward, Duke of Windsor|Edward VIII Edward VIII of the United Kingdom|Edward Windsor|Edward VIII Edward VIII, King of the United Kingdom|Prince Edward|Edward Albert Christian George Andrew Patrick David Windsor|Edward VIII|Prince Edward, Duke of Windsor|Edward, Prince of Wales|Duke of Windsor
Predicted Answer:     King Edward VIII
------------------------------------------------------------
Query [2/5] ID:       infoseek_val_00000075
Question:             What is the closest parent taxonomy of this bird?
Top-1 Entity:         Pomarine jaeger
Ground Truth Answer:  Fulmar|Fulmarus
Predicted Answer:     Great skua
------------------------------------------------------------
Query [3/5] ID:       infoseek_val_00000152
Question:             What is the

### Cell 8: Full 1,000-Query Answer Generation with Checkpoints
Executes answer generation across all 1,000 queries with periodic auto-saving every 50 iterations.


In [8]:
# ---------------------------------------------------------------------------
# Full Generation Loop with Checkpointing
# ---------------------------------------------------------------------------
PREDICTIONS_CHECKPOINT = os.path.join(OUTPUT_DIR, "step3_predictions_checkpoint.json")
FINAL_PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, "step3_predictions.json")
SAVE_INTERVAL = 50

predictions_dict = {}
if os.path.exists(FINAL_PREDICTIONS_FILE):
    print(f"Found completed predictions at {FINAL_PREDICTIONS_FILE}. Loading...")
    with open(FINAL_PREDICTIONS_FILE, "r", encoding="utf-8") as f:
        predictions_dict = json.load(f)
    print(f"Loaded {len(predictions_dict):,} processed queries.")
elif os.path.exists(PREDICTIONS_CHECKPOINT):
    print(f"Found intermediate checkpoint at {PREDICTIONS_CHECKPOINT}. Resuming...")
    with open(PREDICTIONS_CHECKPOINT, "r", encoding="utf-8") as f:
        predictions_dict = json.load(f)
    print(f"Resuming from query {len(predictions_dict):,}...")

all_qids = list(stage2_results.keys())

print(f"Generating answers for {len(all_qids):,} queries...")
for it, qid in tqdm(enumerate(all_qids), total=len(all_qids), desc="Stage 3 Answer Generation"):
    if qid in predictions_dict:
        continue
        
    q_data = stage2_results[qid]
    question = q_data.get("question", "")
    top1_sec = reranked_sections[qid].get("top1_section", "")
    gt_meta = sample_data.get(qid, {})
    
    # Load query image
    img_path = get_query_image_path(qid, QUERY_IMG_DIR, qid_to_img_id)
    if img_path and os.path.isfile(img_path):
        try:
            query_img = PIL.Image.open(img_path).convert("RGB")
        except Exception:
            query_img = PIL.Image.new("RGB", (224, 224), color=(0, 0, 0))
    else:
        query_img = PIL.Image.new("RGB", (224, 224), color=(0, 0, 0))
        
    prompt_text = VLM_INFOSEEK_INSTRUCTION + build_vqa_prompt(question, top1_sec)
    
    try:
        if GENERATOR_TYPE == "llava1_5":
            formatted_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
            inputs = processor(text=formatted_prompt, images=query_img, return_tensors="pt").to(device)
            with torch.no_grad():
                output_tokens = generator_model.generate(
                    **inputs,
                    max_new_tokens=32,
                    do_sample=False
                )
            pred_text = processor.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        else:
            messages = [{"role": "user", "content": prompt_text}]
            enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
            with torch.no_grad():
                output_tokens = generator_model.generate(
                    enc,
                    max_new_tokens=32,
                    do_sample=False
                )
            pred_text = tokenizer.decode(output_tokens[0][enc.shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"Warning: Generation error for {qid}: {e}")
        pred_text = ""
        
    predictions_dict[qid] = {
        "data_id": qid,
        "question": question,
        "prediction": pred_text,
        "ground_truth_answers": gt_meta.get("answer", ""),
        "question_type": gt_meta.get("question_type", ""),
        "ground_truth_url": q_data.get("ground_truth_url", ""),
        "top1_entity_url": q_data.get("top1_entity_url", ""),
        "top1_entity_title": q_data.get("top1_entity_title", ""),
        "top1_section": top1_sec
    }
    
    # Save periodic intermediate checkpoint
    if (it + 1) % SAVE_INTERVAL == 0:
        with open(PREDICTIONS_CHECKPOINT, "w", encoding="utf-8") as f:
            json.dump(predictions_dict, f, indent=2)

# Save final predictions
with open(FINAL_PREDICTIONS_FILE, "w", encoding="utf-8") as f:
    json.dump(predictions_dict, f, indent=2)
print(f"\nSaved final Stage 3 predictions to {FINAL_PREDICTIONS_FILE}")


Generating answers for 1,000 queries...


Stage 3 Answer Generation: 100%|██████████| 1000/1000 [24:14<00:00,  1.45s/it]


Saved final Stage 3 predictions to /kaggle/working/step3_results/step3_predictions.json


### Cell 9: Evaluation Metrics (InfoSeek Standard)
Computes Exact-Match (EM), normalized string match, and numerical/time range accuracy based on OMGM's official evaluation utilities.


In [9]:
import re
import string

def normalize_answer(text: str) -> str:
    """Standard InfoSeek normalization: strips articles, punctuation, whitespace, and lowercases."""
    text = str(text)
    def remove_articles(t):
        return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t):
        return ' '.join(t.split())
    def remove_punc(t):
        return ''.join(ch for ch in t if ch not in set(string.punctuation))
    def lower(t):
        return t.lower()
    return white_space_fix(remove_articles(remove_punc(lower(text))))

def check_match(pred: str, gt_str: str) -> bool:
    """Evaluates prediction against multiple pipe-separated ground truth aliases."""
    norm_pred = normalize_answer(pred)
    if not norm_pred:
        return False
    candidates = [normalize_answer(ans) for ans in str(gt_str).split("|") if ans.strip()]
    for cand in candidates:
        if norm_pred == cand or cand in norm_pred:
            return True
        # Numerical exact substring match
        num_cand = re.findall(r'\d+', cand)
        num_pred = re.findall(r'\d+', norm_pred)
        if num_cand and num_pred and num_cand == num_pred:
            return True
    return False

# Evaluate predictions
total_evaluated = 0
total_correct = 0

for qid, p_info in predictions_dict.items():
    pred = p_info["prediction"]
    gt = p_info["ground_truth_answers"]
    is_correct = check_match(pred, gt)
    p_info["is_correct"] = is_correct
    if is_correct:
        total_correct += 1
    total_evaluated += 1

vqa_acc = (total_correct / max(total_evaluated, 1)) * 100.0
print("=" * 60)
print(f"STAGE 3 VQA EVALUATION RESULTS (N={total_evaluated})")
print("=" * 60)
print(f"Overall VQA Accuracy: {vqa_acc:.2f}% ({total_correct}/{total_evaluated})")
print("=" * 60)


STAGE 3 VQA EVALUATION RESULTS (N=1000)
Overall VQA Accuracy: 30.20% (302/1000)


### Cell 10: Phase 0 Failure Categorization (Research Project Milestone)
Categorizes every question into Success, Category A, B, C, or D per Section 4 of `plan_init.md`.


In [10]:
# ---------------------------------------------------------------------------
# Phase 0 Failure Categorization (Section 4 of Project Plan)
# ---------------------------------------------------------------------------
cat_counts = {
    "Success": 0,
    "Category A": 0,  # Retrieval Ceiling: correct entity never in Top-20
    "Category B": 0,  # Reranking Failure: correct entity was in Top-20, but Stage 2 picked wrong Top-1
    "Category C": 0,  # Section Selection Failure: correct entity picked, but wrong section selected
    "Category D": 0,  # Generation Failure: correct entity + correct section, but generator answered wrong
}

detailed_failures = []

for qid, p_info in predictions_dict.items():
    s2_data = stage2_results[qid]
    gt_url = s2_data.get("ground_truth_url", "").strip()
    top1_url = s2_data.get("top1_entity_url", "").strip()
    cand_urls = [ent["url"].strip() for ent in s2_data.get("retrieved_entities", [])]
    
    is_correct = p_info.get("is_correct", False)
    
    if is_correct:
        cat_counts["Success"] += 1
        cat = "Success"
    else:
        if gt_url not in cand_urls:
            cat = "Category A"
        elif top1_url != gt_url:
            cat = "Category B"
        else:
            # Correct entity was selected! Check section evidence
            top1_sec = p_info.get("top1_section", "")
            gt_ans_list = [normalize_answer(ans) for ans in str(p_info.get("ground_truth_answers", "")).split("|") if ans.strip()]
            
            # Check if ground truth answer exists in selected section
            norm_sec = normalize_answer(top1_sec)
            contains_evidence = any(ans in norm_sec for ans in gt_ans_list)
            
            if contains_evidence:
                cat = "Category D"  # Had evidence in section, but generator hallucinated / failed
            else:
                cat = "Category C"  # Correct article, but selected section lacked the answer
                
        cat_counts[cat] += 1
        
    p_info["failure_category"] = cat
    if cat != "Success":
        detailed_failures.append({
            "data_id": qid,
            "category": cat,
            "question": p_info["question"],
            "prediction": p_info["prediction"],
            "ground_truth": p_info["ground_truth_answers"],
            "top1_entity_url": top1_url,
            "ground_truth_url": gt_url
        })

print("=" * 65)
print("PHASE 0 FAILURE ANALYSIS BREAKDOWN (N=1,000)")
print("=" * 65)
print(f"{'Category':<15} | {'Count':<8} | {'Percentage':<10} | {'Description'}")
print("-" * 65)
total_n = len(predictions_dict)
for cat, cnt in cat_counts.items():
    pct = (cnt / total_n) * 100.0
    desc = {
        "Success": "VQA answer correctly generated",
        "Category A": "Wrong entity never in Stage 1 Top-20",
        "Category B": "Correct entity in Top-20, Stage 2 chose wrong #1",
        "Category C": "Correct entity, but wrong section selected",
        "Category D": "Correct entity & section, generator failed"
    }[cat]
    print(f"{cat:<15} | {cnt:<8} | {pct:6.2f}%    | {desc}")
print("=" * 65)

# Save failure analysis report
REPORT_FILE = os.path.join(OUTPUT_DIR, "phase0_failure_analysis.json")
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "summary": cat_counts,
        "vqa_accuracy": vqa_acc,
        "failures": detailed_failures
    }, f, indent=2)
print(f"Saved Phase 0 Failure Report to: {REPORT_FILE}")


PHASE 0 FAILURE ANALYSIS BREAKDOWN (N=1,000)
Category        | Count    | Percentage | Description
-----------------------------------------------------------------
Success         | 302      |  30.20%    | VQA answer correctly generated
Category A      | 153      |  15.30%    | Wrong entity never in Stage 1 Top-20
Category B      | 205      |  20.50%    | Correct entity in Top-20, Stage 2 chose wrong #1
Category C      | 203      |  20.30%    | Correct entity, but wrong section selected
Category D      | 137      |  13.70%    | Correct entity & section, generator failed
Saved Phase 0 Failure Report to: /kaggle/working/step3_results/phase0_failure_analysis.json
